# 🎬 IW3 2D-to-3D Stereoscopic Video & Image Converter

This Google Colab notebook provides an automated pipeline to convert standard 2D videos and image collections into immersive 3D stereoscopic formats (Side-by-Side, Top-Bottom, VR180) using [`nagadomi/nunif`](https://github.com/nagadomi/nunif)'s **`iw3`** module.

---
### ⚠️ Prerequisites
1. **Hardware Acceleration:** Ensure your Colab runtime is set to GPU (`Runtime` → `Change runtime type` → select `T4 GPU` or higher).
2. **Storage Persistence:** Connecting Google Drive is strongly recommended for video processing to safeguard against runtime disconnects.

In [ ]:
# @title 1. Verify GPU Runtime { display-mode: "form" }
# @markdown Inspect hardware allocation and verify CUDA device availability.

import sys
import shutil
import torch

print("Checking compute environment...")
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not detected! Please go to Runtime -> Change runtime type, "
        "select a GPU accelerator (e.g., T4 GPU), and restart the session."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
total_vram_gb = gpu_props.total_memory / (1024 ** 3)

print(f"✓ Active GPU: {gpu_name}")
print(f"✓ VRAM Total: {total_vram_gb:.2f} GB")
print(f"✓ CUDA Version: {torch.version.cuda}")
print(f"✓ PyTorch Version: {torch.__version__}")

ffmpeg_path = shutil.which("ffmpeg")
if ffmpeg_path:
    print(f"✓ FFmpeg binary detected at: {ffmpeg_path}")
else:
    print("⚠ FFmpeg binary not found in PATH. Video processing requires FFmpeg.")


In [ ]:
# @title 2. Mount Google Drive { display-mode: "form" }
# @markdown Mount Google Drive to read inputs and write outputs without risking timeout data loss.

import os

mount_drive = True # @param {type:"boolean"}
drive_mount_point = "/content/drive" # @param {type:"string"}
default_workspace = "/content/drive/MyDrive/iw3" # @param {type:"string"}

if mount_drive:
    try:
        from google.colab import drive
        if not os.path.ismount(drive_mount_point):
            print(f"Mounting Google Drive to {drive_mount_point}...")
            drive.mount(drive_mount_point)
            print("✓ Google Drive mounted successfully.")
        else:
            print("✓ Google Drive already mounted.")
        
        os.makedirs(default_workspace, exist_ok=True)
        print(f"✓ Workspace directory ready: {default_workspace}")
    except ImportError:
        print("⚠ google.colab module not found (running in local container). Skipping Drive mount.")
        os.makedirs("./workspace", exist_ok=True)
else:
    print("Drive mounting skipped by user parameter. Using local storage.")


In [ ]:
# @title 3. Setup IW3 Environment { display-mode: "form" }
# @markdown Clone nunif repository and install secondary dependencies without altering base PyTorch.

import os
import sys
import subprocess

force_reinstall = False # @param {type:"boolean"}
nunif_dir = "/content/nunif"

# 1. Clone repository
if not os.path.exists(nunif_dir) or force_reinstall:
    if os.path.exists(nunif_dir):
        print("Removing existing nunif directory...")
        subprocess.run(["rm", "-rf", nunif_dir], check=True)
    print("Cloning nagadomi/nunif repository...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/nagadomi/nunif.git", nunif_dir], check=True)
    print("✓ nagadomi/nunif cloned successfully.")
else:
    print(f"✓ Existing nunif repository detected at {nunif_dir}.")

# 2. Add nunif to Python sys.path
if nunif_dir not in sys.path:
    sys.path.insert(0, nunif_dir)

# 3. Install required non-torch dependencies quietly
print("Checking and installing secondary dependencies (timm, packaging, onnxruntime-gpu)...")
install_cmd = [
    sys.executable, "-m", "pip", "install", "-q",
    "timm", "packaging", "onnxruntime-gpu"
]
subprocess.run(install_cmd, check=True)
print("✓ Environment dependencies ready.")

# 4. Verify module availability
try:
    import iw3
    print("✓ iw3 module successfully accessible.")
except ImportError as err:
    raise ImportError(f"Failed to import iw3 from {nunif_dir}: {err}")


In [ ]:
# @title 4. Run 2D to 3D Conversion { display-mode: "form" }
# @markdown Configure source media, stereoscopic layout, and inference parameters.

import os
import sys
import shutil
import subprocess

# --- Storage & Paths ---
input_path = "/content/drive/MyDrive/iw3/input.mp4" # @param {type:"string"}
output_path = "/content/drive/MyDrive/iw3/output_sbs.mp4" # @param {type:"string"}

# --- 3D Stereo Geometry ---
stereo_format = "full-sbs" # @param ["full-sbs", "half-sbs", "tb", "half-tb", "vr180", "anaglyph"]
divergence = 2.0 # @param {type:"slider", min:0.5, max:5.0, step:0.1}
convergence = 0.5 # @param {type:"slider", min:0.0, max:1.0, step:0.05}
method = "forward" # @param ["forward", "row", "cross"]

# --- Model & Performance Tuning ---
depth_model = "Any_V2_L" # @param ["Any_V2_L", "Any_V2_B", "Any_V2_S", "ZoeD_Any_N"]
zoed_height = 384 # @param [320, 384, 512] {type:"raw"}
batch_size = 1 # @param {type:"integer"}
max_fps = 0 # @param {type:"integer"}

# --- Video Encoding & Audio ---
video_codec = "h264_nvenc" # @param ["h264_nvenc", "libx264", "hevc_nvenc", "libx265"]
pix_fmt = "yuv420p" # @param ["yuv420p", "yuv444p"]
keep_audio = True # @param {type:"boolean"}

# --- Execution Engine ---
nunif_dir = "/content/nunif"
if not os.path.exists(nunif_dir):
    raise FileNotFoundError(
        f"nunif directory not found at {nunif_dir}. Please execute Cell 3 first."
    )

if not os.path.exists(input_path):
    raise FileNotFoundError(
        f"Input path does not exist: {input_path}. Please verify the file or directory path."
    )

# Ensure target parent directory exists
out_dir = os.path.dirname(os.path.abspath(output_path))
if out_dir:
    os.makedirs(out_dir, exist_ok=True)

# Build command line arguments
cmd = [
    sys.executable, "-m", "iw3",
    "--input", input_path,
    "--output", output_path,
    "--depth-model", str(depth_model),
    "--stereo-format", str(stereo_format),
    "--divergence", str(float(divergence)),
    "--convergence", str(float(convergence)),
    "--method", str(method),
    "--zoed-height", str(int(zoed_height)),
    "--batch-size", str(int(batch_size)),
    "--video-codec", str(video_codec),
    "--pix-fmt", str(pix_fmt),
]

if int(max_fps) > 0:
    cmd.extend(["--max-fps", str(int(max_fps))])

if not keep_audio:
    cmd.append("--disable-audio")

print("=" * 60)
print("Starting IW3 2D-to-3D Conversion")
print(f"Input:         {input_path}")
print(f"Output:        {output_path}")
print(f"Model:         {depth_model} (Resolution height: {zoed_height})")
print(f"Format:        {stereo_format} (Divergence: {divergence}, Convergence: {convergence})")
print(f"Codec:         {video_codec} (Pixel format: {pix_fmt})")
print("=" * 60)
print(f"Executing: {' '.join(cmd)}\n")

# Stream subprocess execution in real-time
process = subprocess.Popen(
    cmd,
    cwd=nunif_dir,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

output_lines = []
try:
    for line in iter(process.stdout.readline, ''):
        print(line, end='', flush=True)
        output_lines.append(line)
except KeyboardInterrupt:
    process.terminate()
    print("\nProcess interrupted by user.")
    raise

process.wait()

if process.returncode != 0:
    full_log = "".join(output_lines)
    if "CUDA out of memory" in full_log:
        print("\n" + "!" * 60)
        print("ERROR: CUDA Out of Memory (OOM) encountered!")
        print("Remediation options:")
        print("  1. Reduce 'zoed_height' to 320 in the form above.")
        print("  2. Change 'depth_model' to 'Any_V2_B' or 'Any_V2_S'.")
        print("  3. Ensure 'batch_size' is set to 1.")
        print("!" * 60)
    raise RuntimeError(f"iw3 conversion failed with exit code {process.returncode}")

print("\n" + "=" * 60)
print(f"✓ Conversion completed successfully!")
print(f"✓ Output saved to: {output_path}")
print("=" * 60)


In [ ]:
# @title 5. Verify Output Stream { display-mode: "form" }
# TODO: Phase 3 will implement stream geometry assertions via ffprobe and stream integrity null-mux checks via ffmpeg.
print("Cell 5: Verification stub. Scheduled for Phase 3.")
